In [1]:
# from ollama import chat

# # Initialize an empty message history
# messages = []
# while True:
#     user_input = input('Chat with history: ')
#     if user_input.lower() == 'exit':
#         break
#     # Get streaming response while maintaining conversation history
#     response_content = ""
#     for chunk in chat(
#         'deepseek-r1:1.5b',
#         messages=messages + [
#             {'role': 'system', 'content': 'You are a helpful assistant. You only give a short sentence by answer.'},
#             {'role': 'user', 'content': user_input},
#         ],
#         stream=True
#     ):
#         if chunk.message:
#             response_chunk = chunk.message.content
#             print(response_chunk, end='', flush=True)
#             response_content += response_chunk
#     # Add the exchange to the conversation history
#     messages += [
#         {'role': 'user', 'content': user_input},
#         {'role': 'assistant', 'content': response_content},
#     ]
#     print('\n')  # Add space after response

In [2]:
# Configuration des chemins
DATA_FOLDER = "data"
files = [
    "Convention collective - services alimentaires.pdf",
    "Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf",
    "Membres du comité SST.pdf",
    "politiques_hotel_de_la_promenade_document.pdf",
    "Principes de vie internes.pdf",
    "Processus de Gestion des reservations.pdf"
]

def get_document_category(filename):
    """Assigne une catégorie basée sur le nom du fichier pour le filtrage RAG."""
    if "Convention" in filename: return "RH - Légal"
    if "politiques" in filename: return "Opérations - Manuel"
    if "SST" in filename: return "Sécurité - Santé"
    if "Critères" in filename: return "Qualité - Service"
    return "Interne - Culture"


In [3]:
import os
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text):
    # 1. On protège les accents avant toute manipulation si nécessaire
    # Mais ici on va surtout nettoyer les espaces doubles et les sauts de ligne incohérents
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 2. Supprimer les en-têtes répétitifs (Hôtel de la Promenade) 
    # seulement s'ils sont seuls sur une ligne
    lines = text.split('\n')
    cleaned_lines = [l for l in lines if "PROMENADE" not in l.upper() or len(l) > 30]
    text = "\n".join(cleaned_lines)
    
    # 3. Recoller les mots coupés par un saut de ligne (ex: "pré-\namble")
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    
    return text.strip()

In [4]:
def get_section_title(text):
    """Détermine dynamiquement la section du chunk."""
    # Cherche "ARTICLE X", "CHAPITRE X" ou "Annexe X"
    match = re.search(r'(ARTICLE\s+\d+|CHAPITRE\s+\d+|ANNEXE\s+[A-Z])', text, re.IGNORECASE)
    if match:
        return match.group(0).upper()
    return None

In [5]:
from langchain_core.documents import Document

def process_hotel_docs(files):
    all_chunks = []
    
    # Splitter pour les gros documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\nARTICLE ", "\nCHAPITRE ", "\n\d+\. ", "\n\n", "\n", ". "],
        keep_separator=True
    )

    for file_name in files:
        file_path = f"data/{file_name}"
        loader = PyPDFLoader(file_path)
        pages = loader.load()
        
        # 1. On regroupe tout le texte du fichier pour évaluer sa taille
        full_text = "\n".join([page.page_content for page in pages])
        cleaned_content = clean_text(full_text)
        
        category = get_document_category(file_name)
        
        # --- LOGIQUE OPTION B : TAILLE CONDITIONNELLE ---
        # Si le document est petit (ex: moins de 2000 caractères), on ne le coupe pas.
        # Cela garantit que l'adresse email et les membres du SST restent ensemble.
        if len(cleaned_content) < 2000:
            metadata = {
                "source": file_name,
                "categorie": category,
                "section": "Document Complet",
                "page": "1+",
                "type": "petit_doc_integral"
            }
            # On injecte le titre du document au début du texte pour le retriever
            final_content = f"DOCUMENT: {file_name}\n{cleaned_content}"
            all_chunks.append(Document(page_content=final_content, metadata=metadata))
            print(f"Traitement : {file_name} gardé en un seul bloc.")
            
        else:
            # Pour les gros documents (Convention, Politiques), on découpe normalement
            print(f"Traitement : {file_name} découpé en chunks.")
            chunks = text_splitter.split_text(cleaned_content)
            
            last_section = "Introduction"
            for i, chunk in enumerate(chunks):
                section = get_section_title(chunk)
                if section: last_section = section
                
                metadata = {
                    "source": file_name,
                    "categorie": category,
                    "section": last_section,
                    "page": "Calculée", # On peut affiner ici
                    "type": "chunk_sequence"
                }
                
                # Injection de contexte : on force le nom du doc et la section dans le texte
                contextualized_content = f"SOURCE: {file_name} | SECTION: {last_section}\n{chunk}"
                
                all_chunks.append(Document(page_content=contextualized_content, metadata=metadata))
                
    return all_chunks

In [6]:
rag_ready_data = process_hotel_docs(files)
print(f"Nombre total de chunks : {len(rag_ready_data)}")
print(f"Exemple : {rag_ready_data[2].page_content[:200]}")
print(f"Métadonnées : {rag_ready_data[2].metadata}")

Traitement : Convention collective - services alimentaires.pdf découpé en chunks.
Traitement : Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf gardé en un seul bloc.
Traitement : Membres du comité SST.pdf gardé en un seul bloc.
Traitement : politiques_hotel_de_la_promenade_document.pdf découpé en chunks.
Traitement : Principes de vie internes.pdf découpé en chunks.
Traitement : Processus de Gestion des reservations.pdf gardé en un seul bloc.
Nombre total de chunks : 89
Exemple : SOURCE: Convention collective - services alimentaires.pdf | SECTION: ARTICLE 1
ARTICLE 1 - PRÉAMBULE 
1.1 Objectifs de la présente entente 
La présente convention collective a pour but : 
a) D’établir
Métadonnées : {'source': 'Convention collective - services alimentaires.pdf', 'categorie': 'RH - Légal', 'section': 'ARTICLE 1', 'page': 'Calculée', 'type': 'chunk_sequence'}


In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. Plus besoin de boucle de conversion ! 
# rag_ready_data contient déjà des objets Document avec page_content et metadata.
documents_for_chroma = rag_ready_data 

# 2. Configuration des embeddings
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

# 3. Création du vectorstore avec persistance
# J'ajoute le dossier de persistance pour éviter de recalculer à chaque lancement
vectorstore = Chroma.from_documents(
    documents=documents_for_chroma, 
    embedding=embeddings_model
)

# 4. Configuration du retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,          # On récupère un peu plus de morceaux
        "fetch_k": 15,   # On pioche dans un plus grand réservoir initial
        "lambda_mult": 0.6 # Augmente la diversité des sources trouvées
    }
)

In [8]:
# On affine le prompt pour le personnel de l'hôtel
SYSTEM_PROMPT = """Tu es l'Assistant RH et Opérations de l'Hôtel de la Promenade.
Ton rôle est d'aider les employés à trouver des informations fiables dans les documents internes.

RÈGLES DE RÉPONSE :
1. Utilise UNIQUEMENT le CONTEXTE fourni.
2. Cite TOUJOURS la source (Nom du fichier et Page) à la fin de ta réponse.
3. Si l'information concerne une procédure, donne les étapes clairement (liste à puces).
4. Si l'information n'est pas dans le contexte, réponds : "Je suis désolé, je ne trouve pas cette information dans les manuels officiels. Veuillez contacter votre superviseur ou les RH."
5. Reste professionnel, bienveillant et factuel.
"""

In [9]:
from ollama import chat

def format_docs_with_metadata(docs):
    """
    Formate les chunks de manière structurée pour optimiser la réponse du LLM
    en incluant la hiérarchie des sections.
    """
    if not docs:
        return "AUCUN CONTEXTE DISPONIBLE."

    context_blocks = []
    
    for i, doc in enumerate(docs, 1):
        # Extraction sécurisée des métadonnées
        source = doc.metadata.get('source', 'Document inconnu')
        page = doc.metadata.get('page', 'N/A')
        cat = doc.metadata.get('categorie', 'Information générale')
        section = doc.metadata.get('section', 'Section non spécifiée')
        
        # Formatage du bloc avec une structure claire
        block = (
            f"--- EXTRAIT {i} ---\n"
            f"PROVENANCE : {source}\n"
            f"CONTEXTE : {cat} | {section}\n"
            f"EMPLACEMENT : Page {page}\n"
            f"CONTENU : {doc.page_content.strip()}\n"
            f"----------------"
        )
        context_blocks.append(block)
    
    return "\n\n".join(context_blocks)

In [10]:
# Initialisation
messages = []

while True:
    user_input = input("\nQuestion Employé > ").strip()
    if not user_input: continue
    if user_input.lower() in ["exit", "quitter"]: break

    # 1. Récupération des chunks (On s'assure que le retriever répond)
    retrieved_docs = retriever.invoke(user_input)
    
    if not retrieved_docs:
        print("Assistant: Je n'ai trouvé aucune information officielle à ce sujet.")
        continue

    # 2. Formatage du contexte
    context_text = format_docs_with_metadata(retrieved_docs)

    # 3. Prompt système mis à jour pour CHAQUE question
    # On place le contexte dans le 'system' pour que le LLM le traite en priorité
    current_system_prompt = f"{SYSTEM_PROMPT}\n\nCONTEXTE :\n{context_text}"

    # Construction du message (On garde un historique court pour la fluidité)
    api_messages = [{'role': 'system', 'content': current_system_prompt}]
    for msg in messages[-4:]: # Ajout de l'historique conversationnel
        api_messages.append(msg)
    api_messages.append({'role': 'user', 'content': user_input})

    print(f"Utilisateur: {user_input}")
    print(f"Assistant: ", end="", flush=True)
    
    full_response = ""
    
    # 4. Boucle de streaming corrigée
    try:
        response_generator = chat(model="lfm2.5-thinking:1.2b-q8_0", messages=api_messages, stream=True)
        
        for chunk in response_generator:
            # Correction : Accès direct au contenu du message
            content = chunk['message']['content']
            
            if content:
                # Filtrage optionnel des balises de pensée
                clean_content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
                if "<think>" not in content: # Évite d'afficher le début d'une balise
                    print(clean_content, end="", flush=True)
                full_response += content

    except Exception as e:
        print(f"\nErreur de connexion avec Ollama : {e}")

    print("\n" + "-"*50)

    # 5. Archivage de la réponse (sans les balises think)
    final_answer = re.sub(r'<think>.*?</think>', '', full_response, flags=re.DOTALL).strip()
    
    if final_answer:
        messages.append({"role": "user", "content": user_input})
        messages.append({"role": "assistant", "content": final_answer})

Utilisateur: c'est quoi un grief ?
Assistant: Un grief est une divergence d'opinion ou d'interprétation concernant une interprétation, une application ou une violation de la convention collective, comme définie dans l'article 8. Ce concept est ancré dans le cadre de la Convention collective - services alimentaires.pdf (Page 9). Il peut être résolu par des étapes spécifiques, selon les circonstances.  
Source : Convention collective - services alimentaires.pdf (Page 9).  

Citez-vous la source ci-dessus.
--------------------------------------------------
Utilisateur: C'est quoi la mission, la vision et les valeurs de l'hotel ?
Assistant: La **mission** de l'hôtel se concentre sur son rôle central dans la satisfaction des clients et la qualité de l'expérience globale. La **vision** définit son objectif à long terme, visant à devenir une référence dans la ville pour la qualité et l'accueil. Les **valeurs** guident ses actions, reflétant des principes comme la durabilité, l'intégrité et l'